<a href="https://colab.research.google.com/github/rohith154-vsr/ML-2-LAB/blob/main/LP_4_CE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# load car evaluation dataset (from local CSV or download from UCI repository)
csv_file = "CAR_EVALUATION.csv"
if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
else:
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data"
    column_names = ["buying", "maint", "doors", "persons", "lug_boot", "safety", "Target"]
    df = pd.read_csv(url, names=column_names)

print("Dataset shape:", df.shape)
print("Class counts:
", df["Target"].value_counts())

# convert to binary classification: 'good' and 'vgood' are positive, others negative
df["Class"] = df["Target"].apply(
    lambda x: "Positive" if x in ["good", "vgood"] else "Negative"
)

features = ["buying", "maint", "doors", "persons", "lug_boot", "safety"]

# select a balanced subset of 8 positive and 7 negative instances for clean demonstration
positive = df[df["Class"] == "Positive"]
negative = df[df["Class"] == "Negative"]

selected_rows = []
for i in range(8):
    selected_rows.append(positive.iloc[i])
for i in range(7):
    selected_rows.append(negative.iloc[i])

sample_df = pd.DataFrame(selected_rows).sample(frac=1, random_state=42).reset_index(drop=True)
print("
Selected 15 mixed examples:")
print(sample_df[["Target", "Class"] + features])

# check whether hypothesis h covers example x (all non-? values must match)
def covers(hypothesis, example):
    for h, x in zip(hypothesis, example):
        if h == "A~":
            return False
        if h != "?" and h != x:
            return False
    return True

# returns True if h1 is strictly more general than h2
def is_more_general(h1, h2):
    more_general = False
    for a, b in zip(h1, h2):
        if a == "?" and b != "?":
            more_general = True
        elif a != "?" and b == "?":
            return False
        elif a != b and a != "?" and b != "?":
            return False
    return more_general

# check if general hypothesis g is consistent with at least one hypothesis in S
def is_consistent_with_S(g, S):
    return any(is_more_general(g, s) or g == s for s in S)

# Candidate-Elimination algorithm: maintains dual boundaries S (specific) and G (general)
def candidate_elimination(sample_df, features):
    n = len(features)
    # S initialized to most specific, G initialized to most general
    S = [["A~"] * n]
    G = [["?"] * n]
    s_history = [S[0][:]]
    g_history = [G[0][:]]

    for _, row in sample_df.iterrows():
        example = [row[f] for f in features]
        label = row["Class"]

        if label == "Positive":
            # positive instance: prune G hypotheses that do not cover this example
            G = [g for g in G if covers(g, example)]
            
            # generalize S to cover this positive instance
            new_S = []
            for s in S:
                if covers(s, example):
                    new_S.append(s)
                else:
                    # generalize minimally: replace mismatches with '?'
                    generalized = []
                    for i in range(n):
                        if s[i] == "A~":
                            gen = example[:]
                        else:
                            gen = s[:]
                            if gen[i] != example[i]:
                                gen[i] = "?"
                        # keep only generalizations that remain consistent with G
                        if any(is_more_general(g, gen) or g == gen for g in G):
                            generalized.append(gen)
                    new_S.extend(generalized)
            
            # prune redundant hypotheses: keep only the most specific in S
            S = []
            for s in new_S:
                if not any(s != t and is_more_general(s, t) for t in new_S):
                    if s not in S:
                        S.append(s)
            if not S:
                S = [example[:]]

        else: # Negative instance
            # negative instance: prune S hypotheses that cover this negative example
            S = [s for s in S if not covers(s, example)]
            
            # specialize G to exclude this negative instance
            new_G = []
            for g in G:
                if not covers(g, example):
                    new_G.append(g)
                else:
                    # specialize minimally by replacing '?' with attribute values that differ from example
                    for i in range(n):
                        if g[i] == "?":
                            vals = sample_df[features[i]].unique()
                            for v in vals:
                                if v != example[i]:
                                    spec = g[:]
                                    spec[i] = v
                                    # keep only specializations that are more general than some hypothesis in S
                                    if any(is_more_general(spec, s) or spec == s for s in S):
                                        new_G.append(spec)
            
            # prune redundant hypotheses: keep only the most general in G
            G = []
            for g in new_G:
                if not any(g != h and is_more_general(h, g) for h in new_G):
                    if g not in G:
                        G.append(g)
            if not G:
                G = [["?"] * n]

        s_history.append(S[0][:] if S else ["A~"] * n)
        g_history.append(G[0][:] if G else ["?"] * n)

    return S, G, s_history, g_history

S_final, G_final, s_hist, g_hist = candidate_elimination(sample_df, features)

print("
Final Specific Boundary S:")
for s in S_final:
    print(" ", dict(zip(features, s)))

print("
Final General Boundary G:")
for g in G_final:
    print(" ", dict(zip(features, g)))

# plot boundary evolution heatmap (0 = specific 'A~', 0.5 = value, 1 = general '?')
import numpy as np

def boundary_to_numeric(history, n_features):
    matrix = []
    for h in history:
        row = []
        for v in h:
            if v == "A~":
                row.append(0)
            elif v == "?":
                row.append(1)
            else:
                row.append(0.5)
        matrix.append(row)
    return np.array(matrix)

s_matrix = boundary_to_numeric(s_hist, len(features))
g_matrix = boundary_to_numeric(g_hist, len(features))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(s_matrix, cmap="YlOrRd", aspect="auto")
axes[0].set_title("S Boundary Evolution (Specific -> Generalized)")
axes[0].set_xticks(range(len(features)))
axes[0].set_xticklabels(features, rotation=45)
axes[0].set_ylabel("Step")

axes[1].imshow(g_matrix, cmap="Blues", aspect="auto")
axes[1].set_title("G Boundary Evolution (General -> Specialized)")
axes[1].set_xticks(range(len(features)))
axes[1].set_xticklabels(features, rotation=45)
axes[1].set_ylabel("Step")

plt.tight_layout()
plt.show()